Исправляет ситуацию, когда в одной строке панели содержиться информация по нескольким подотраслям. Равномерно распределяет исходное значение на итоговые подотрасли.

In [21]:
import pandas as pd

df_main = pd.read_excel('original.xlsx', dtype={'subindustry_code': str, 'OKVED': str, 'year': str})
df_map = pd.read_excel('mapping.xlsx', dtype={'subindustry_code': str, 'OKVED': str, 'year': str})

# ====== 1. считаем число OKVED на subindustry_code ======
okved_cnt = (
    df_map
    .groupby('subindustry_code')
    .size()
    .rename('n_okved')
    .reset_index()
)

# ====== 2. добавляем счётчик в основную таблицу ======
df = df_main.merge(okved_cnt, on='subindustry_code', how='left')

df['n_okved'] = df['n_okved'].fillna(1).astype(int)

# ====== 3. расширяем строки по мэппингу ======
# OKVED из df_map перезапишет OKVED из df_main
df = df.merge(
    df_map,
    on='subindustry_code',
    how='left',
    suffixes=('', '_map')
)

# ====== 4. заменяем OKVED ======
df['OKVED'] = df['OKVED_map'].fillna(df['OKVED'])
df = df.drop(columns='OKVED_map')

# ====== 5. делим int-поля ======
int_cols = df.select_dtypes(include='int').columns.tolist()
int_cols = [c for c in int_cols if c not in ['n_okved']]

for col in int_cols:
    df[col] = df[col] / df['n_okved']

# ====== 6. убираем служебную колонку ======
df_final = df.drop(columns='n_okved')


# ====== 7. сохраняем ======
df_final.to_excel('panel_expanded.xlsx', index=False)